# Cross-Linguistic Generalization of Syntactic Attention in Transformers
## Experimental Pipeline — Final Version

This notebook implements the full experimental pipeline used in the article
*Cross-Linguistic Generalization of Syntactic Attention in Transformers: Analysis of Romance Languages*.

**Notebook roadmap**
- §1 — Environment setup and validation
- §2 — Attention extraction (UD × monolingual models)
- §3 — Generalization metrics
- §4 — Negative control (German effect)
- §5 — Pairwise language correlation analysis
- §6 — Visualizations (layer profiles, heatmaps, Spearman matrices)
- §7 — Entropy by language
- §8 — Stability across splits
- §9 — Exported files for the paper

The notebook is organized so that each section answers one practical question:
what was run, how it was measured, and which files support the final paper.

---
## §1 — Environment setup and validation

This section checks whether the output folders exist, whether GPU is available,
and whether all required scripts are present before running the pipeline.
---

In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path
import pandas as pd
import numpy as np
import torch

# Directories
BASE_DIR = Path("./output")
METRICS_MONO = BASE_DIR / "metrics_mono"
FIGURAS_DIR  = BASE_DIR / "figuras"

for d in [BASE_DIR, METRICS_MONO, FIGURAS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# GPU check
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CPU — slower execution")

# Required scripts
scripts = [
    "ud_attention_eval_core.py",
    "run_ud_attention_eval.py",
    "compute_generalization_metrics.py",
    "lang_resources.py",
]
for s in scripts:
    print(f"{'OK' if Path(s).exists() else 'MISSING'}  {s}")


GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Memory: 5.7 GB
MISSING  ud_attention_eval_core.py
MISSING  run_ud_attention_eval.py
MISSING  compute_generalization_metrics.py
MISSING  lang_resources.py


---
## §2 — Attention extraction

We process sentences from 7 UD treebanks using independent monolingual BERT-like models.
For each syntactic arc (	extit{nsubj}, 	extit{obj}, 	extit{case}, 	extit{amod}),
we compute mean attention between head and dependent tokens across all
144 positions (12 layers × 12 heads).

**Main settings**
- Languages: pt, gl, es, fr, it, ro + de (control)
- Splits: train, dev, test
- Models: one monolingual model per language (no mBERT in the final analysis)
- Sentences: 1000 per split
- Maximum length: 512 subwords
- Relations: nsubj, obj, amod, case
---

In [4]:
OUT_CSV = BASE_DIR / "attention_all_splits.csv"

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

# Monolingual + mBERT extraction (uncomment to run)
# !python run_ud_attention_eval.py \
#   --langs pt,gl,es,it,fr,ro \
#   --include_control --control_langs de \
#   --splits train,dev,test \
#   --model_mode mono_vs_mbert \
#   --max_sents 1000 \
#   --max_len 512 \
#   --rel_filter nsubj,obj,amod,case \
#   --out_csv {OUT_CSV.as_posix()}

# Inspection
df_raw = pd.read_csv(OUT_CSV)
print(f"Shape: {df_raw.shape}")
print(f"Languages: {sorted(df_raw['lang'].unique())}")
print(f"Families: {sorted(df_raw['model_family'].unique())}")
print(f"\nRows per (lang, model_family):")
print(df_raw.groupby(['lang', 'model_family']).size().to_string())

# Final analysis: monolingual models only
df = df_raw[df_raw["model_family"] == "mono"].copy()
print(f"\nMonolingual-only shape: {df.shape}")


GPU available: True
Device: NVIDIA GeForce RTX 3050 6GB Laptop GPU


FileNotFoundError: [Errno 2] No such file or directory: 'output/attention_all_splits.csv'

### §2.1 — Arc counts by language, relation, and split

Before interpreting the results, we check whether each relation has enough arcs
in each language and split. This is a basic data-coverage sanity check.

In [ ]:
arcs = (
    df[(df["layer"] == 1) & (df["head"] == 1) & (df["direction"] == "head_to_dep")]
    .groupby(["lang", "deprel", "split"])["n_arcs"]
    .sum()
    .reset_index()
)
piv = arcs.pivot_table(index=["lang", "split"], columns="deprel", values="n_arcs", aggfunc="sum")
print(piv.to_string())

---
## §3 — Generalization metrics (monolingual models)

We compute Spearman correlation between attention profiles of language pairs
at two complementary levels:
- **Micro** ($\rho_{\text{micro}}$): 144-position vectors (layer × head)
- **Macro** ($\rho_{\text{macro}}$): 12-position vectors (mean by layer)

We also compute normalized entropy (concentration vs. spread)
and a composite stability score.
---

In [ ]:
!python compute_generalization_metrics.py \
  --in_csv {OUT_CSV.as_posix()} \
  --direction head_to_dep \
  --out_dir {METRICS_MONO.as_posix()}

m = pd.read_csv(METRICS_MONO / "generalization_metrics_by_deprel.csv")
print("=== GENERALIZATION METRICS (monolingual models) ===\n")
print(m.sort_values(["split", "stability_score_0_1"], ascending=[True, False]).to_string(index=False))


---
## §4 — Negative control: German effect

Here we compare metrics computed only among Romance languages (6 languages)
with metrics computed after adding German (7 languages).
If correlation drops after adding German, this suggests a typological effect.
---

In [ ]:
# Romance-only subset
ROMANCE = ["pt", "gl", "es", "it", "fr", "ro"]
df_rom = df[df["lang"].isin(ROMANCE)].copy()
rom_csv = BASE_DIR / "attention_romance_only.csv"
df_rom.to_csv(rom_csv, index=False)

METRICS_ROM = BASE_DIR / "metrics_romance_only"
METRICS_ROM.mkdir(exist_ok=True)

!python compute_generalization_metrics.py \
  --in_csv {rom_csv.as_posix()} \
  --direction head_to_dep \
  --out_dir {METRICS_ROM.as_posix()}

# Comparison
m_rom = pd.read_csv(METRICS_ROM / "generalization_metrics_by_deprel.csv")
m_rom = m_rom[m_rom["split"] == "(todos)"][["deprel", "spearman_wmean", "spearman_mean_layer"]].copy()
m_rom.columns = ["deprel", "micro_ROM", "macro_ROM"]

m_all = m[m["split"] == "(todos)"][["deprel", "spearman_wmean", "spearman_mean_layer"]].copy()
m_all.columns = ["deprel", "micro_ALL", "macro_ALL"]

cmp = m_rom.merge(m_all, on="deprel")
cmp["delta_micro"] = cmp["micro_ALL"] - cmp["micro_ROM"]
cmp["delta_macro"] = cmp["macro_ALL"] - cmp["macro_ROM"]
print(cmp.to_string(index=False))


---
## §5 — Pairwise Spearman correlation across languages

We build the full correlation matrix for all 7 languages,
for each dependency relation. This helps identify which language pairs
show the strongest convergence in attention profiles.
---

In [ ]:
from itertools import combinations
from scipy.stats import spearmanr

def build_vector_144(sub):
    """144-position vector (12 layers × 12 heads) for a given (lang, deprel)."""
    v = np.zeros(144)
    for _, row in sub.iterrows():
        idx = (int(row["layer"]) - 1) * 12 + (int(row["head"]) - 1)
        v[idx] = float(row["mean_attention"])
    return v

df_h2d = df[df["direction"] == "head_to_dep"].copy()
langs = sorted(df_h2d["lang"].unique())
deprels = sorted(df_h2d["deprel"].unique())

# Compute one matrix per dependency relation
for dep in deprels:
    vecs = {}
    for lang in langs:
        sub = df_h2d[(df_h2d["deprel"] == dep) & (df_h2d["lang"] == lang)]
        if not sub.empty:
            vecs[lang] = build_vector_144(sub)

    n = len(langs)
    print(f"\n=== {dep} ===")
    header = "     " + " ".join([f"{x:>7s}" for x in langs])
    print(header)
    for i, li in enumerate(langs):
        rowvals = []
        for j, lj in enumerate(langs):
            if i == j:
                rho = 1.0
            elif li in vecs and lj in vecs:
                rho, _ = spearmanr(vecs[li], vecs[lj])
            else:
                rho = np.nan
            rowvals.append(f"{rho:7.2f}" if pd.notna(rho) else "   nan ")
        print(f"{li:>3s}  " + " ".join(rowvals))


---
## §6 — Visualizations

### §6.1 — Layer-wise attention profiles (Romance languages vs. German)

These figures summarize where syntactic attention tends to emerge across layers.
The blue line is the Romance mean, the shaded band shows min--max variation,
and the dashed red line corresponds to German.
---

### §6.2 — Heatmaps (layer × head) by relation

One heatmap is produced for each dependency relation.
These files are saved separately so they can be arranged later in a 2×2 article layout.

In [ ]:
for dep in deprels:
    sub = df_h2d[df_h2d["deprel"] == dep]
    piv = sub.pivot_table(index="layer", columns="head", values="mean_attention", aggfunc="mean")
    layers = sorted(piv.index)
    heads = sorted(piv.columns)
    mat = piv.reindex(index=layers, columns=heads).values

    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(mat, aspect="auto", cmap="YlOrRd", origin="lower")
    ax.set_xlabel("Head", fontsize=11)
    ax.set_ylabel("Layer", fontsize=11)
    ax.set_title(f"{dep}", fontsize=13, fontweight="bold")
    ax.set_xticks(range(len(heads)))
    ax.set_xticklabels(heads, fontsize=8)
    ax.set_yticks(range(len(layers)))
    ax.set_yticklabels(layers, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()

    fig.savefig(FIGURAS_DIR / f"heatmap_{dep}.pdf")
    fig.savefig(FIGURAS_DIR / f"heatmap_{dep}.png", dpi=150)
    plt.show()
    print(f"[saved] heatmap_{dep}.pdf")

In [ ]:
for dep in deprels:
    vecs = {}
    for lang in langs:
        sub = df_h2d[(df_h2d["deprel"] == dep) & (df_h2d["lang"] == lang)]
        if not sub.empty:
            vecs[lang] = build_vector_144(sub)

    n = len(langs)
    mat = np.ones((n, n))
    for i in range(n):
        for j in range(n):
            if i != j and langs[i] in vecs and langs[j] in vecs:
                rho, _ = spearmanr(vecs[langs[i]], vecs[langs[j]])
                mat[i, j] = rho

    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(mat, cmap="RdYlGn", vmin=-0.2, vmax=1.0)
    ax.set_xticks(range(n))
    ax.set_xticklabels(langs, fontsize=10)
    ax.set_yticks(range(n))
    ax.set_yticklabels(langs, fontsize=10)
    ax.set_title(f"{dep}", fontsize=13, fontweight="bold")

    for i in range(n):
        for j in range(n):
            color = "white" if mat[i, j] < 0.4 else "black"
            ax.text(j, i, f"{mat[i,j]:.2f}", ha="center", va="center", fontsize=8, color=color)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()

    fig.savefig(FIGURAS_DIR / f"spearman_{dep}.pdf")
    fig.savefig(FIGURAS_DIR / f"spearman_{dep}.png", dpi=150)
    plt.show()
    print(f"[saved] spearman_{dep}.pdf")

### §6.3 — Spearman correlation matrices across languages

One matrix is produced for each dependency relation.
These files are also saved separately for later use in a 2×2 article layout.

---
## §7 — Normalized entropy by language

Normalized entropy measures how attention is distributed across the 144 positions.
Values close to 1 indicate a more uniform distribution;
values close to 0 indicate concentration in a small number of heads.
---

In [ ]:
m_all = pd.read_csv(METRICS_MONO / "generalization_metrics_by_deprel.csv")
splits_only = m_all[m_all["split"] != "(todos)"].copy()

print("=== METRICS BY SPLIT ===\n")
for dep in sorted(splits_only["deprel"].unique()):
    print(f"--- {dep} ---")
    sub = splits_only[splits_only["deprel"] == dep].sort_values("split")
    for _, r in sub.iterrows():
        print(f"  {r['split']:20s}  rho_micro={r['spearman_wmean']:.4f}  rho_macro={r['spearman_mean_layer']:.4f}  S={r['stability_score_0_1']:.4f}")
    print()

print("=== STANDARD DEVIATION ACROSS SPLITS ===\n")
var = splits_only.groupby("deprel").agg(
    micro_std=("spearman_wmean", "std"),
    macro_std=("spearman_mean_layer", "std"),
    stab_std=("stability_score_0_1", "std"),
).round(4)
print(var.to_string())


---
## §8 — Stability across splits (train / dev / test)

Here we verify whether the metrics remain stable across treebank splits.
Low standard deviation suggests that the results are not mere sampling artifacts.
---

In [ ]:
import os

print("=== GENERATED FILES ===\n")
for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(str(BASE_DIR), "").count(os.sep)
    indent = "  " * level
    folder = os.path.basename(root)
    print(f"{indent}{folder}/")
    subindent = "  " * (level + 1)
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        if size > 1024*1024:
            sstr = f"{size/1024/1024:.1f} MB"
        elif size > 1024:
            sstr = f"{size/1024:.0f} KB"
        else:
            sstr = f"{size} B"
        print(f"{subindent}{f:50s} {sstr:>8s}")


---
## §9 — Summary of generated files

All outputs are saved under `output_pipeline_corrigido/`.
This final check is useful before writing the paper or committing the project to Git.
---

---
## Mapping to the paper

| Paper table / figure | Generated file | Notebook section |
|---|---|---|
| Table 1 — Global metrics | `metrics_mono/generalization_metrics_by_deprel.csv` | §3 |
| Table 2 — Metrics by split | same file (column `split`) | §8 |
| Table 3 — Entropy by language | `metrics_mono/entropy_by_lang.csv` | §7 |
| Figures 2–5 — Layer profiles | `figuras/layer_profile_*_mono_en.pdf` | §6.1 |
| Figure 6 — Heatmaps (a)–(d) | `figuras/heatmap_*_en.pdf` | §6.2 |
| Figure 1 — Spearman matrices | `figuras/spearman_*_en.pdf` | §6.3 |
---

In [ ]:
"""
Extension: Bootstrap + permutation for statistical significance
Requires: attention_mono_only.csv (output of the main pipeline)
"""

import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import spearmanr

# Configuration
CSV_PATH = "output_pipeline_corrigido/attention_mono_only.csv"
N_BOOT = 1000
N_PERM = 1000
CI = 0.95
SEED = 42

# Load and filter
df = pd.read_csv(CSV_PATH)
df = df[df["direction"] == "head_to_dep"].copy()

langs = sorted(df["lang"].unique())
deprels = sorted(df["deprel"].unique())

def build_vector(sub):
    v = np.zeros(144)
    for _, row in sub.iterrows():
        idx = (int(row["layer"]) - 1) * 12 + (int(row["head"]) - 1)
        v[idx] = float(row["mean_attention"])
    return v

def weighted_micro(df_local, dep):
    vecs = {}
    weights = {}
    for lang in langs:
        sub = df_local[(df_local["lang"] == lang) & (df_local["deprel"] == dep)]
        if sub.empty:
            continue
        vecs[lang] = build_vector(sub)
        weights[lang] = sub["n_arcs"].sum()

    rhos, ws = [], []
    for l1, l2 in combinations(vecs.keys(), 2):
        rho, _ = spearmanr(vecs[l1], vecs[l2])
        if np.isfinite(rho):
            rhos.append(rho)
            ws.append(min(weights[l1], weights[l2]))
    if not rhos:
        return np.nan
    return np.average(rhos, weights=ws)

rng = np.random.default_rng(SEED)
results = []

for dep in deprels:
    observed = weighted_micro(df, dep)

    # Bootstrap
    boot_vals = []
    dep_df = df[df["deprel"] == dep]
    for _ in range(N_BOOT):
        parts = []
        for lang in langs:
            sub = dep_df[dep_df["lang"] == lang]
            if sub.empty:
                continue
            idx = rng.integers(0, len(sub), len(sub))
            parts.append(sub.iloc[idx])
        if parts:
            boot_df = pd.concat(parts, axis=0)
            boot_vals.append(weighted_micro(boot_df, dep))
    boot_vals = np.array([x for x in boot_vals if np.isfinite(x)])
    lo = np.quantile(boot_vals, (1 - CI) / 2)
    hi = np.quantile(boot_vals, 1 - (1 - CI) / 2)

    # Permutation
    perm_vals = []
    for _ in range(N_PERM):
        perm_df = dep_df.copy()
        perm_df["mean_attention"] = rng.permutation(perm_df["mean_attention"].values)
        perm_vals.append(weighted_micro(perm_df, dep))
    perm_vals = np.array([x for x in perm_vals if np.isfinite(x)])

    mu0 = perm_vals.mean()
    sd0 = perm_vals.std(ddof=1)
    z = (observed - mu0) / sd0 if sd0 > 0 else np.nan
    p = (np.sum(perm_vals >= observed) + 1) / (len(perm_vals) + 1)

    results.append({
        "deprel": dep,
        "rho_micro": observed,
        "ci_low": lo,
        "ci_high": hi,
        "perm_mean": mu0,
        "perm_sd": sd0,
        "z_score": z,
        "p_value": p,
    })

res = pd.DataFrame(results).sort_values("rho_micro", ascending=False)
print(res.round(4).to_string(index=False))
